In [ ]:
pip install scikit-learn==1.4.2 imbalanced-learn==0.12.2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

In [ ]:
df=pd.read_csv('SMLC_ESA_MODEL.csv')
df_copy=df.copy()
df.head()

In [ ]:
df.shape
print(f"rows: {df.shape[0]} columns: {df.shape[1]}")

In [ ]:
df.info()



In [ ]:
df.head()

In [ ]:
cat_cols = ['Education','City','Gender','EverBenched']
df[cat_cols] = df[cat_cols].astype('category') #
df['LeaveOrNot']=df['LeaveOrNot'].astype('category') # No encoding required. COnverting to category for analysis purpose
df['JoinedSince'] = (pd.Timestamp.today().year-df['JoiningYear']).astype('float64')
df=df.drop('JoiningYear',axis=1)

#final cat and numerical columns
cat_cols = df.select_dtypes(include='category').columns
numeric_cols = df.select_dtypes(include=['float64','int64']).columns



In [ ]:

print(df[numeric_cols].describe())
print(df[cat_cols].describe())

# from the values
# Age and EverBenched   having missing values



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
for col in cat_cols:
    print(f'Count of unique categories {col}','\n',df[col].nunique())
    print(f'Count of values in each {col} categories','\n',df[col].value_counts())

#EverBenched   having missing values

In [ ]:
target = 'LeaveOrNot'
for col in numeric_cols:
    if col!=target:
        sns.violinplot(x=df[col],y=df[target])
        plt.xlabel(col)
        plt.ylabel(target)
        plt.show()

for col in cat_cols:
    if col!=target:
        sns.countplot(data=df,x=col,order = df[col].value_counts().sort_values(ascending=True).index,hue=target)
        plt.xlabel(col)
        plt.show()

#Target variable distribution
sns.countplot(data=df,x='LeaveOrNot',order = df['LeaveOrNot'].value_counts().sort_values(ascending=True).index,hue='LeaveOrNot')
plt.xlabel('LeaveOrNot')
plt.show()


    


In [ ]:
df.duplicated().sum()
print('Missing value counts','\n',df.isnull().sum())

sns.heatmap(df.isnull())
plt.title('Missing Values')
plt.show()




In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['EverBenched'] = df['EverBenched'].fillna('No')

In [ ]:
plt.figure(figsize=(15,10))
for idx,col in enumerate(numeric_cols,1):
    plt.subplot(3,3,idx)
    sns.boxplot(df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

plt.figure(figsize=(15,10))
for idx,col in enumerate(cat_cols,1):
    plt.subplot(3,3,idx)
    sns.boxplot(df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

#Valid data in paymenttier, education and everbenched. Hence no outlier treatment is required. We will do the encoding once aftre the data cleanup

In [ ]:
df['LeaveOrNot'].value_counts(normalize=True)
#this shows moderate imbalance. No treatment required. We need to finely tune this during data model

In [ ]:
print(df.head())
print(df.info())
#Education, City --> n-1 dummy
#Gender,EnverBenched - Label

le=LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender']).astype('float64')
df['EverBenched'] = le.fit_transform(df['EverBenched']).astype('float64')
df['LeaveOrNot'] = df['LeaveOrNot'].astype('float64')



In [ ]:
df.info()

In [ ]:

df=pd.get_dummies(df,columns=['Education','City'],drop_first=True)

In [ ]:
sns.heatmap(df[numeric_cols].corr(),annot=True,cmap='coolwarm')
plt.title('Correlation heat map')
plt.show()

In [ ]:
print (numeric_cols)

In [ ]:
df.corr(numeric_only=True)['LeaveOrNot'].sort_values(ascending=False)

In [ ]:
bool_cols = df.select_dtypes(include = 'bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
num_cols = ['Age','ExperienceInCurrentDomain',
            'WorkHoursPerWeek','AnnualBonus','JoinedSince']
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3-Q1 

    lower = Q1 - 1.5*IQR 
    upper = Q3 + 1.5*IQR

    outlier = df[(df[col] < lower) | (df[col] >upper)][col].count()

    print(f"{col} : {outlier} outliers")

In [ ]:
scaler = StandardScaler()
X = df.drop('LeaveOrNot', axis = 1)
y = df['LeaveOrNot']
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=43,stratify=y)
X_train[num_cols]=scaler.fit_transform(X_train[num_cols])
X_test[num_cols]=scaler.transform(X_test[num_cols])

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
lr = LogisticRegression()
logit_model = lr.fit(X_train,y_train)
y_prob = logit_model.predict_proba(X_test)[:,1]
thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    y_pred = (y_prob>= t).astype(int)
    
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)

    results.append([t,f1,precision,recall])
metric_df = pd.DataFrame(results, columns= ['Thresholds', 'F1 score', 'Precision','Recall'])
print(metric_df)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight = 'balanced', random_state=42)
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state = 42)

models = ({'Logistic Regression': lr, 'Decision Tree': dt, "Random Forest":rf})

results = []

for name, model in models.items():
    model.fit(X_train,y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    results.append([name,
    accuracy_score(y_test,y_pred), recall_score(y_test,y_pred),precision_score(y_test,y_pred),f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_pred)])
    
final_df = pd.DataFrame(results, columns=['Name','Accuracy_score','recall','precission','f1','roc Auc'])
print(final_df)

In [ ]:
rf_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=15,
    min_samples_leaf=5,
    random_state=42
)
rf_tuned.fit(X_train, y_train)

y_pred = rf_tuned.predict(X_test)
y_prob = rf_tuned.predict_proba(X_test)[:, 1]

rf_results = pd.DataFrame([[
    accuracy_score(y_test, y_pred),
    precision_score(y_test, y_pred),
    recall_score(y_test, y_pred),
    f1_score(y_test, y_pred),
    roc_auc_score(y_test, y_prob)
]], columns=["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"])

print(rf_results)

In [ ]:
importances = rf.feature_importances_
fi_df = pd.DataFrame({'Feature':X_train.columns, 'Importance':importances}).sort_values(by='Importance', ascending=False)
print(fi_df)

plt.figure(figsize=(8,5))
plt.barh(fi_df['Feature'],fi_df['Importance'])
plt.title("Feature Importance - Random Forest")
plt.show()